# Data Profiling Report
## Tableau Prep Cleaning Exercise

Pre-cleaning data profiling only. No transformations are applied here.

### Contents
1. Load & Overview
2. Missing Value Analysis
3. Data Type Analysis
4. Duplicate Detection
5. Cardinality Analysis
6. Value Distribution Analysis
7. Text Format & Inconsistency Analysis
8. Date Format Analysis
9. Numeric Validity & Outlier Analysis
10. Column Analysis (Identifiers, Patterns, Domains)
11. Structure Analysis (Functional Dependencies, Calculated Fields)
12. Cleaning Rules Summary

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("data/tableau_prep_cleaning_exercise.csv")
df.shape

(100, 18)

In [2]:
# Preview first 10 rows
df.head(10)

,Order_ID,Customer_ID,Customer_Name,Email,Country,City,Order_Date,Ship_Date,Product_Category,Quantity,Unit_Price,Discount,Sales,Age,Customer_Segment,Satisfaction_Score,Payment_Method,Notes
0,O1001,C016,MAHA ZADEH,maha.zadeh@example.com,Deutschland,Mannheim,2026/01/27,not available,office supplies,2,19.73,0.25,29.59,60.0,Corporate,5.0,Bank Transfer,ok
1,O1002,C025,Sara Smith,sara.smith@example.com,germany,karlsruhe,03-30-2026,2026-04-08,electronics,7,285.86,0.10,1800.92,52.0,CORPORATE,NaN,Bank Transfer,ok
2,O1003,C014,Carlos Garcia,carlos.garcia@example.com,Italy,Rome,19/01/2026,2026/01/28,Office Supplies,7,312.17,0.10,1966.67,69.0,consumer,1.0,NaN,check address
3,O1004,C009,Nina Weber,nina.weber@example.com,Deutschland,Mannheim,2026/05/18,22.05.2026,Home,4,226.13,0.00,904.52,25.0,Corporate,4.0,Credit Card,return risk
4,O1005,C018,Maha Zadeh,maha.zadeh@example.com,italy,Naples,29.01.2026,02/02/2026,Electronics,12,86.96,0.25,782.64,35.0,Home Office,2.0,bank transfer,check address
5,O1006,C054,CARLOS GARCIA,carlos.garcia@example.com,italy,Naples,2026/01/15,2026-01-18,Furniture,2,63.59,0.20,101.74,26.0,home office,4.0,Bank Transfer,missing phone
6,O1007,C034,Nina Weber,nina.weber@example.com,Spain,Madrid,02-27-2026,2026-02-28,home,0,34.02,0.00,0.00,71.0,CORPORATE,3.0,PAYPAL,check address
7,O1008,C043,Julia Fischer,julia.fischer@example.com,FRANCE,Lyon,2026-04-16,2026-04-23,Home,2,206.29,0.10,371.32,45.0,NaN,4.0,Credit Card,missing phone
8,O1009,C032,John Doe,john.doe@example.com,Spain,Madrid,NaN,13.04.2026,Office Supplies,8,330.88,0.25,1985.28,61.0,Consumer,2.0,Credit Card,missing phone
9,O1010,C004,Tom Brown,tom.brown@example.com,spain,Barcelona,2026/02/17,2026-02-18,furniture,2,263.88,0.00,527.76,42.0,CORPORATE,3.0,PayPal,ok


---
## 1. Load & Overview

In [3]:
print(df.dtypes)
df.describe()

Order_ID                  str
Customer_ID               str
Customer_Name             str
Email                     str
Country                   str
City                      str
Order_Date                str
Ship_Date                 str
Product_Category          str
Quantity                int64
Unit_Price                str
Discount              float64
Sales                 float64
Age                   float64
Customer_Segment          str
Satisfaction_Score    float64
Payment_Method            str
Notes                     str
dtype: object


,Quantity,Discount,Sales,Age,Satisfaction_Score
count,100.00000,100.000000,100.000000,93.000000,85.000000
mean,5.77000,0.130500,2477.780800,48.494624,3.117647
std,3.33895,0.077491,7144.014096,19.671003,1.247968
min,0.00000,0.000000,0.000000,16.000000,1.000000
25%,3.00000,0.050000,376.577500,35.000000,2.000000
50%,6.00000,0.150000,1246.685000,46.000000,3.000000
75%,8.00000,0.200000,2012.237500,60.000000,4.000000
max,12.00000,0.250000,56999.940000,130.000000,5.000000


## Findings

`df.dtypes` reveals that `Unit_Price` was inferred as `object` rather than a numeric type,
indicating the presence of non-numeric characters in that column.
Both `Order_Date` and `Ship_Date` are typed as `object`, confirming that no consistent
date format was detected by the parser.
All remaining columns were inferred at expected types.

`df.describe()` reports counts below 100 for `Age` (93) and `Satisfaction_Score` (85),
indicating 7 and 15 missing values respectively without requiring an explicit null check.
`Sales` shows a mean of 2,477.78 against a median of 1,246.69 and a maximum of 56,999.94,
suggesting right skew driven by high-value outliers.
`Age` has a maximum of 130, which falls outside the range of plausible human age values.
`Quantity` has a minimum of 0, indicating the presence of orders with no items recorded.

---
## 2. Missing Value Analysis

We check for two types of missingness:
- **True NaN** — pandas reads the cell as null
- **Pseudo-null strings** — values like `"not available"`, `"N/A"`, `""` that are not null but carry no information

In [4]:
true_nulls = df.isnull().sum()
true_null_pct = (df.isnull().mean() * 100).round(2)

null_summary = pd.DataFrame({
    "Missing (NaN)": true_nulls,
    "Missing %": true_null_pct
})

null_summary[null_summary["Missing (NaN)"] > 0].sort_values("Missing %", ascending=False)

,Missing (NaN),Missing %
Satisfaction_Score,15,15.0
Payment_Method,14,14.0
Customer_Segment,12,12.0
Email,10,10.0
Notes,10,10.0
Age,7,7.0
Order_Date,5,5.0
City,3,3.0


In [5]:
PSEUDO_NULLS = {"not available", "n/a", "na", "none", "null", "-", "unknown", ""}

for col in df.select_dtypes(include="str").columns:
    mask = df[col].str.strip().str.lower().isin(PSEUDO_NULLS)
    if mask.any():
        found = df.loc[mask, col].unique().tolist()
        print(f"{col}: {found}")

Ship_Date: ['not available']


### Missing Value Findings

| Column | NaN Count | Type |
|---|---|---|
| `Satisfaction_Score` | 15 | True NaN |
| `Payment_Method` | 14 | True NaN |
| `Customer_Segment` | 12 | True NaN |
| `Email` | 10 | True NaN |
| `Notes` | 10 | True NaN |
| `Age` | 7 | True NaN |
| `Order_Date` | 5 | True NaN |
| `City` | 3 | True NaN |
| `Ship_Date` | 0 | Pseudo-null string `"not available"` in 2 rows, not detected by standard null checks |

---
## 3. Data Type Analysis

In [6]:
df.dtypes

Order_ID                  str
Customer_ID               str
Customer_Name             str
Email                     str
Country                   str
City                      str
Order_Date                str
Ship_Date                 str
Product_Category          str
Quantity                int64
Unit_Price                str
Discount              float64
Sales                 float64
Age                   float64
Customer_Segment          str
Satisfaction_Score    float64
Payment_Method            str
Notes                     str
dtype: object

In [7]:
def is_clean_number(val):
    if pd.isna(val):
        return True
    try:
        float(str(val).strip())
        return True
    except ValueError:
        return False

df[~df["Unit_Price"].apply(is_clean_number)][["Order_ID", "Customer_Name", "Unit_Price"]]

,Order_ID,Customer_Name,Unit_Price
11,O1012,John Doe,"9999,99"
12,O1013,Luca Bianchi,€282.33
21,O1022,JULIA FISCHER,€275.80
28,O1029,NINA WEBER,€215.16
34,O1035,TOM BROWN,€137.67
38,O1039,anna müller,"412,54"
55,O1056,Ali Reza,€74.09
56,O1057,John Doe,"239,38"
64,O1065,luca bianchi,€291.72
74,O1075,Julia Fischer,"9999,99"


### Data Type Findings

| Column | Expected Type | Actual Type | Problem |
|---|---|---|---|
| `Unit_Price` | Float | String | 6 rows with `€` prefix (e.g. `€282.33`); 6 rows with comma as decimal separator (e.g. `9999,99`) |
| `Order_Date` | Date | String | Multiple date formats detected, no single parse possible |
| `Ship_Date` | Date | String | Multiple date formats detected; pseudo-null `"not available"` confirmed in section 2 |
| `Age` | Integer | Float | Coerced to float by pandas due to NaN rows; maximum value of 130 visible in `describe()` |

---
## 4. Duplicate Detection

In [8]:
dupes = df[df.duplicated(subset=["Order_ID"], keep=False)].sort_values("Order_ID")
print(f"Duplicate Order_IDs: {df['Order_ID'].duplicated().sum()}")
print(f"Full row duplicates: {df.duplicated(keep=False).sum()}")
dupes[["Order_ID", "Customer_ID", "Customer_Name", "Order_Date", "Sales"]]

Duplicate Order_IDs: 5
Full row duplicates: 10


,Order_ID,Customer_ID,Customer_Name,Order_Date,Sales
4,O1005,C018,Maha Zadeh,29.01.2026,782.64
95,O1005,C018,Maha Zadeh,29.01.2026,782.64
17,O1018,C049,MEHDI KARIMI,28/01/2026,2604.13
96,O1018,C049,MEHDI KARIMI,28/01/2026,2604.13
28,O1029,C010,NINA WEBER,2026-02-06,1828.86
97,O1029,C010,NINA WEBER,2026-02-06,1828.86
47,O1048,C046,Carlos Garcia,01.04.2026,444.45
98,O1048,C046,Carlos Garcia,01.04.2026,444.45
63,O1064,C034,Nina Weber,11/02/2026,1371.33
99,O1064,C034,Nina Weber,11/02/2026,1371.33


### Duplicate Findings

Five Order_IDs appear exactly twice: O1005, O1018, O1029, O1048, O1064.
In each case the duplicate row is identical across all 18 columns, confirmed by the
full row duplicate count of 10 (5 pairs). The duplicates are located at rows 95 to 99,
suggesting they originate from a repeated data export rather than a data entry error.

---
## 5. Cardinality Analysis

In [9]:
pd.DataFrame({
    "Unique": df.nunique(),
    "Total": len(df),
    "% Unique": (df.nunique() / len(df) * 100).round(1)
})

,Unique,Total,% Unique
Order_ID,95,100,95.0
Customer_ID,46,100,46.0
Customer_Name,39,100,39.0
Email,15,100,15.0
Country,15,100,15.0
City,15,100,15.0
Order_Date,82,100,82.0
Ship_Date,87,100,87.0
Product_Category,10,100,10.0
Quantity,13,100,13.0


`Order_ID` has 95 unique values across 100 rows. The 5 missing correspond to the duplicate rows identified in section 4.
`Email` has 15 distinct values across 46 unique Customer_IDs.

---
## 6. Value Distribution Analysis

In [10]:
EXCLUDE_COLS = ["Order_ID", "Customer_ID", "Customer_Name", "Email",
                "Order_Date", "Ship_Date", "Unit_Price"]

CAT_COLS = [col for col in df.select_dtypes(include="str").columns
            if col not in EXCLUDE_COLS]

for col in CAT_COLS:
    print(df[col].value_counts(dropna=False).rename(col))

Country
Deutschland    10
Germany        10
Italy           9
Spain           9
ITALY           8
Netherlands     8
france          7
italy           6
FRANCE          6
NETHERLANDS     6
GERMANY         6
spain           5
germany         4
France          4
España          2
Name: Country, dtype: int64
City
Mannheim      10
Karlsruhe     10
Rome           9
Madrid         9
Milan          8
Amsterdam      7
Naples         6
Lyon           6
paris          6
Heidelberg     6
Barcelona      5
Rotterdam      5
karlsruhe      4
Paris          4
NaN            3
Valencia       2
Name: City, dtype: int64
Product_Category
Office Supplies    19
home               12
furniture          12
electronics        10
Home               10
Electronics        10
ELECTRONICS         8
office supplies     7
Furniture           6
OFFICE SUPPLIES     6
Name: Product_Category, dtype: int64
Customer_Segment
CORPORATE      23
home office    16
Consumer       16
Home Office    13
NaN            12
Corporate  

In [11]:
# Observed range in describe() is 1 to 5
df[df["Satisfaction_Score"].notna() &
   ~df["Satisfaction_Score"].isin([1, 2, 3, 4, 5])][["Order_ID", "Satisfaction_Score"]]

,Order_ID,Satisfaction_Score


In [12]:
df[df["Quantity"] == 0][["Order_ID", "Customer_Name", "Product_Category", "Quantity", "Unit_Price", "Sales"]]

,Order_ID,Customer_Name,Product_Category,Quantity,Unit_Price,Sales
6,O1007,Nina Weber,home,0,34.02,0.0
34,O1035,TOM BROWN,home,0,€137.67,0.0
55,O1056,Ali Reza,home,0,€74.09,0.0
69,O1070,PETER SCHMIDT,Office Supplies,0,74.93,0.0


### Distribution Findings

**Categorical columns**

`Country` contains 15 raw unique values. Variants of the same country appear in
uppercase, lowercase, and title case. Two values use non-English names:
`Deutschland` (Germany) and `España` (Spain).

`City` contains 16 raw unique values including 3 NaN. Casing variants are present
for at least two cities (`paris`/`Paris`, `karlsruhe`/`Karlsruhe`).

`Product_Category` contains 10 raw unique values representing 4 distinct categories:
`electronics`, `furniture`, `home`, and `office supplies`. All variation is due to casing.

`Customer_Segment` contains 6 raw unique values representing 3 distinct segments:
`corporate`, `consumer`, and `home office`. All variation is due to casing.

`Payment_Method` contains 7 raw unique values representing 4 distinct methods:
`bank transfer`, `cash`, `credit card`, and `paypal`. All variation is due to casing.

`Notes` contains 5 distinct values with no casing inconsistency detected.

**Domain check**

No values outside the observed range of 1 to 5 were found in `Satisfaction_Score`.

**Quantity = 0**

Four rows have `Quantity = 0` with `Sales = 0.0` in all four cases. Two of the four
rows also contain dirty `Unit_Price` values.

---
## 7. Text Format & Inconsistency Analysis

In [13]:
# Leading/trailing whitespace in Customer_Name
df[df["Customer_Name"].str.strip() != df["Customer_Name"]][["Order_ID", "Customer_Name"]]

,Order_ID,Customer_Name
7,O1008,Julia Fischer
20,O1021,Emma Wilson
41,O1042,john doe
48,O1049,Mehdi Karimi


In [14]:
# Casing variants grouped by normalised name
name_groups = (df.groupby(df["Customer_Name"].str.strip().str.lower())["Customer_Name"]
                 .unique())
name_groups.index.name = "Normalised"
name_groups[name_groups.apply(len) > 1].reset_index()

,Normalised,Customer_Name
0,ali reza,"[Ali Reza, ali reza]"
1,anna müller,"[ANNA MÜLLER, anna müller, Anna Müller]"
2,carlos garcia,"[Carlos Garcia, CARLOS GARCIA, carlos garcia]"
3,emma wilson,"[Emma Wilson, Emma Wilson ]"
4,john doe,"[John Doe, john doe , JOHN DOE]"
5,julia fischer,"[ Julia Fischer , Julia Fischer, JULIA FISCHER]"
6,laura rossi,"[Laura Rossi, LAURA ROSSI, laura rossi]"
7,luca bianchi,"[Luca Bianchi, luca bianchi]"
8,maha zadeh,"[MAHA ZADEH, Maha Zadeh]"
9,marie dubois,"[MARIE DUBOIS, Marie Dubois, marie dubois]"


In [15]:
EMAIL_PATTERN = r'^[\w.+-]+@[\w-]+\.[\w.]+$'

df[df["Email"].notna() &
   ~df["Email"].str.match(EMAIL_PATTERN)][["Order_ID", "Customer_Name", "Email"]]

,Order_ID,Customer_Name,Email


### Text Format Findings

`Customer_Name` contains leading and trailing whitespace in 4 rows.
Casing variants of the same name exist across 14 normalised names, including
all-caps, title case, and all-lowercase representations of the same person.

`City`, `Product_Category`, `Customer_Segment`, and `Payment_Method` contain
casing variants of the same values, as shown in the distribution section.

`Email` format validation found no invalid patterns among non-null values.

---
## 8. Date Format Analysis

In [16]:
def detect_date_format(val):
    if pd.isna(val) or str(val).strip().lower() in ["not available", ""]:
        return "null/pseudo-null"
    val = str(val).strip()
    patterns = [
        (r'^\d{4}/\d{2}/\d{2}$',   'YYYY/MM/DD'),
        (r'^\d{4}-\d{2}-\d{2}$',   'YYYY-MM-DD'),
        (r'^\d{2}/\d{2}/\d{4}$',   'DD/MM/YYYY'),
        (r'^\d{2}-\d{2}-\d{4}$',   'MM-DD-YYYY'),
        (r'^\d{2}\.\d{2}\.\d{4}$', 'DD.MM.YYYY'),
    ]
    for pat, label in patterns:
        if re.match(pat, val):
            return label
    return f'unknown: {val}'

pd.DataFrame({col: df[col].apply(detect_date_format).value_counts(dropna=False)
              for col in ["Order_Date", "Ship_Date"]})

,Order_Date,Ship_Date
DD.MM.YYYY,21,17
DD/MM/YYYY,23,22
MM-DD-YYYY,18,17
YYYY-MM-DD,19,22
YYYY/MM/DD,14,20
null/pseudo-null,5,2


In [17]:
def try_parse_date(val):
    if pd.isna(val) or str(val).strip().lower() in ["not available", ""]:
        return pd.NaT
    for fmt in ["%Y/%m/%d", "%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d.%m.%Y"]:
        try:
            return pd.to_datetime(str(val).strip(), format=fmt)
        except ValueError:
            continue
    return pd.NaT

order_dates = df["Order_Date"].apply(try_parse_date)
ship_dates  = df["Ship_Date"].apply(try_parse_date)

bad_seq = ship_dates.notna() & order_dates.notna() & (ship_dates < order_dates)
df.loc[bad_seq, ["Order_ID", "Customer_Name", "Order_Date", "Ship_Date"]]

,Order_ID,Customer_Name,Order_Date,Ship_Date


In [18]:
same_day = ship_dates.notna() & order_dates.notna() & (ship_dates == order_dates)
df.loc[same_day, ["Order_ID", "Customer_Name", "Order_Date", "Ship_Date"]]

,Order_ID,Customer_Name,Order_Date,Ship_Date
13,O1014,Julia Fischer,08.04.2026,08/04/2026
40,O1041,laura rossi,01-06-2026,06/01/2026


### Date Format Findings

Both `Order_Date` and `Ship_Date` contain five distinct date formats across their values:
`YYYY/MM/DD`, `YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`, and `DD.MM.YYYY`.
Both columns also contain missing values, documented in section 2.

The hyphen-separated format (`DD-MM-YYYY` vs `MM-DD-YYYY`) is ambiguous in isolation.
Values such as `01-06-2026` cannot be resolved without assumptions about format consistency
across rows. The parser applied here assumes `MM-DD-YYYY` for all hyphen-separated values.

No rows were found where `Ship_Date` preceded `Order_Date`. Two rows show identical
order and ship dates. One of the two (`O1041`) is subject to the format ambiguity above.

---
## 9. Numeric Validity & Outlier Analysis

In [19]:
df[(df["Age"] > 120) | (df["Age"] < 0)][["Order_ID", "Customer_Name", "Age"]]

,Order_ID,Customer_Name,Age
47,O1048,Carlos Garcia,130.0
98,O1048,Carlos Garcia,130.0


In [20]:
df[df["Unit_Price"].str.contains("9999", na=False)][["Order_ID", "Customer_Name", "Unit_Price", "Quantity"]]

,Order_ID,Customer_Name,Unit_Price,Quantity
11,O1012,John Doe,"9999,99",1
13,O1014,Julia Fischer,9999.99,6
16,O1017,Maha Zadeh,9999.99,4
74,O1075,Julia Fischer,"9999,99",3


In [21]:
Q1, Q3 = df["Sales"].quantile(0.25), df["Sales"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print(f"Lower fence: {lower:.2f} | Upper fence: {upper:.2f}")
df[(df["Sales"] < lower) | (df["Sales"] > upper)][["Order_ID", "Customer_Name", "Unit_Price", "Quantity", "Discount", "Sales"]]

Lower fence: -2076.91 | Upper fence: 4465.73


,Order_ID,Customer_Name,Unit_Price,Quantity,Discount,Sales
11,O1012,John Doe,"9999,99",1,0.05,9499.99
13,O1014,Julia Fischer,9999.99,6,0.05,56999.94
16,O1017,Maha Zadeh,9999.99,4,0.05,37999.96
74,O1075,Julia Fischer,"9999,99",3,0.15,25499.97


### Numeric Validity Findings

`Age` contains one value of 130, which falls outside the plausible human range.
It appears twice due to the duplicate row identified in section 4, both corresponding
to Order_ID O1048.

`Unit_Price` contains four rows where the value is 9999.99 or 9999,99. This value
appears exclusively with a discount of 0.05 and is consistent across unrelated
customers and product categories, suggesting it is a sentinel placeholder rather
than a real price.

The IQR outlier detection on `Sales` identified the same four rows. All Sales outliers
are fully explained by the 9999.99 Unit_Price values and no independent Sales anomalies
were detected.

---
## 10. Structure Analysis

In [22]:
norm_counts = df.groupby("Customer_ID")["Customer_Name"].apply(
    lambda x: x.str.strip().str.lower().nunique()
)
violations = norm_counts[norm_counts > 1].index
df[df["Customer_ID"].isin(violations)].groupby("Customer_ID")["Customer_Name"].unique()

Customer_ID
C003                          [Luca Bianchi, Laura Rossi]
C004                            [Tom Brown, marie dubois]
C005                [Maha Zadeh, Anna Müller, nina weber]
C006                         [MARIE DUBOIS, luca bianchi]
C009                           [Nina Weber, Luca Bianchi]
C010                           [NINA WEBER, Marie Dubois]
C012                              [Anna Müller, Ali Reza]
C014            [Carlos Garcia, Marie Dubois, nina weber]
C017                         [Emma Wilson, carlos garcia]
C018                          [Maha Zadeh, CARLOS GARCIA]
C019    [John Doe, Ali Reza, TOM BROWN, marie dubois, ...
C020                       [Maha Zadeh,   Mehdi Karimi  ]
C021                            [mehdi karimi, tom brown]
C022               [carlos garcia, Anna Müller, John Doe]
C024                             [John Doe, mehdi karimi]
C029     [John Doe, Anna Müller, Julia Fischer, JOHN DOE]
C030                            [laura rossi, Nina Weber]
C0

In [23]:
def clean_price(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace('€','').replace('$','').replace('£','').replace(',','.')
    try:
        return float(s)
    except:
        return np.nan

price_clean = df["Unit_Price"].apply(clean_price)
expected_sales = (price_clean * df["Quantity"] * (1 - df["Discount"])).round(2)
diff = (df["Sales"] - expected_sales).abs()

inconsistent = df[diff > 0.05].copy()
inconsistent["expected_sales"] = expected_sales[diff > 0.05]
inconsistent[["Order_ID", "Unit_Price", "Quantity", "Discount", "expected_sales", "Sales"]]

,Order_ID,Unit_Price,Quantity,Discount,expected_sales,Sales


### Structure Analysis Findings

The functional dependency `Customer_ID → Customer_Name` was checked by grouping
all orders by `Customer_ID` and counting the number of distinct normalised names
per ID. 31 of the 46 unique Customer_IDs map to more than one distinct person.

Representative examples include C019, which maps to five different individuals,
and C054, which maps to four. The violations are not attributable to casing
inconsistency, as names were normalised before comparison.

`Customer_ID` cannot be treated as a reliable customer identifier in this dataset.

The calculated field `Sales` was validated against the formula
`Unit_Price × Quantity × (1 − Discount)` after cleaning the Unit_Price column.
No inconsistencies were found. All Sales values are fully explained by their
corresponding Unit_Price, Quantity, and Discount values.

## Problems Found

| # | Column(s) | Problem |
|---|---|---|
| 1 | All | 5 exact duplicate rows: O1005, O1018, O1029, O1048, O1064 |
| 2 | `Satisfaction_Score` | 15 true NaN |
| 3 | `Payment_Method` | 14 true NaN |
| 4 | `Customer_Segment` | 12 true NaN |
| 5 | `Email` | 10 true NaN |
| 6 | `Notes` | 10 true NaN |
| 7 | `Age` | 7 true NaN |
| 8 | `Order_Date` | 5 true NaN |
| 9 | `City` | 3 true NaN |
| 10 | `Ship_Date` | Pseudo-null string `"not available"` in 2 rows |
| 11 | `Unit_Price` | Parsed as string due to dirty values |
| 12 | `Unit_Price` | `€` prefix in 6 rows |
| 13 | `Unit_Price` | Comma as decimal separator in 6 rows |
| 14 | `Unit_Price` | Placeholder value `9999.99` / `9999,99` in 4 rows |
| 15 | `Order_Date`, `Ship_Date` | 5 mixed date formats across both columns |
| 16 | `Order_Date`, `Ship_Date` | Hyphen format ambiguous: MM-DD vs DD-MM indistinguishable in isolation |
| 17 | `Customer_Name` | Mixed casing across 14 normalised names |
| 18 | `Customer_Name` | Leading and trailing whitespace in 4 rows |
| 19 | `Country` | Mixed casing variants |
| 20 | `Country` | Non-English names: `Deutschland` and `España` |
| 21 | `City` | Mixed casing variants |
| 22 | `Product_Category` | Mixed casing variants |
| 23 | `Customer_Segment` | Mixed casing variants |
| 24 | `Payment_Method` | Mixed casing variants |
| 25 | `Age` | Value of 130 outside plausible human range |
| 26 | `Quantity` | 4 rows with value 0, resulting in Sales of 0.0 |
| 27 | `Sales` | 4 outlier values driven entirely by `Unit_Price` placeholder |
| 28 | `Customer_ID` | 31 of 46 IDs map to more than one distinct person |

## Cleaning Steps Applied in Tableau Prep

| # | Step | Column | Formula / Action |
|---|---|---|---|
| 1 | Remove duplicates | All | Aggregate step: all 18 columns in Grouped Fields |
| 2 | Remove zero quantity | Quantity | `[Quantity] != 0` |
| 3 | Fix Unit_Price | Unit_Price | `IF REPLACE(REPLACE([Unit_Price], '€', ''), ',', '.') = '9999.99' THEN NULL ELSE FLOAT(REPLACE(REPLACE([Unit_Price], '€', ''), ',', '.')) END` |
| 4 | Fix Order_Date | Order_Date | `IF ISNULL([Order_Date]) THEN NULL ELSEIF MID([Order_Date], 5, 1) = '-' THEN DATEPARSE('yyyy-MM-dd', [Order_Date]) ELSEIF MID([Order_Date], 5, 1) = '/' THEN DATEPARSE('yyyy/MM/dd', [Order_Date]) ELSEIF CONTAINS([Order_Date], '.') THEN DATEPARSE('dd.MM.yyyy', [Order_Date]) ELSEIF CONTAINS([Order_Date], '-') THEN DATEPARSE('MM-dd-yyyy', [Order_Date]) ELSEIF CONTAINS([Order_Date], '/') THEN DATEPARSE('dd/MM/yyyy', [Order_Date]) ELSE NULL END` |
| 5 | Fix Ship_Date | Ship_Date | `IF ISNULL([Ship_Date]) THEN NULL ELSEIF [Ship_Date] = 'not available' THEN NULL ELSEIF MID([Ship_Date], 5, 1) = '-' THEN DATEPARSE('yyyy-MM-dd', [Ship_Date]) ELSEIF MID([Ship_Date], 5, 1) = '/' THEN DATEPARSE('yyyy/MM/dd', [Ship_Date]) ELSEIF CONTAINS([Ship_Date], '.') THEN DATEPARSE('dd.MM.yyyy', [Ship_Date]) ELSEIF CONTAINS([Ship_Date], '-') THEN DATEPARSE('MM-dd-yyyy', [Ship_Date]) ELSEIF CONTAINS([Ship_Date], '/') THEN DATEPARSE('dd/MM/yyyy', [Ship_Date]) ELSE NULL END` |
| 6 | Trim and title case | Customer_Name | `PROPER(TRIM([Customer_Name]))` |
| 7 | Replace and title case | Country | `PROPER(REPLACE(REPLACE(TRIM([Country]), 'Deutschland', 'Germany'), 'España', 'Spain'))` |
| 8 | Trim and title case | City | `PROPER(TRIM([City]))` |
| 9 | Trim and title case | Product_Category | `PROPER(TRIM([Product_Category]))` |
| 10 | Trim and title case | Customer_Segment | `PROPER(TRIM([Customer_Segment]))` |
| 11 | Trim and title case | Payment_Method | `PROPER(TRIM([Payment_Method]))` |
| 12 | Remove impossible value | Age | `IF [Age] = 130 THEN NULL ELSE [Age] END` |
| 13 | Recalculate | Sales | `ROUND(([Unit_Price] * [Quantity] * (1 - [Discount])), 2)` |
| 14 | Impute nulls | Age | `IF ISNULL([Age]) THEN 45 ELSE [Age] END` |
| 15 | Impute nulls | Satisfaction_Score | `IF ISNULL([Satisfaction_Score]) THEN 3 ELSE [Satisfaction_Score] END` |
| 16 | Impute nulls | Payment_Method | `IF ISNULL([Payment_Method]) THEN 'Bank Transfer' ELSE [Payment_Method] END` |
| 17 | Impute nulls | Customer_Segment | `IF ISNULL([Customer_Segment]) THEN 'Corporate' ELSE [Customer_Segment] END` |
| 18 | Replace nulls | Notes | `IF ISNULL([Notes]) THEN 'none' ELSE [Notes] END` |
| 19 | No action | Email, City, Order_Date | Nulls retained |
| 20 | Flag only | Customer_ID | Unreliable identifier, no fix applied |